# Kesiapan penelitian dan pembagian data

Notebook ini merangkum bukti kelayakan IDSSE, mengunci pembagian
train/validation/test pada tingkat pertandingan, dan menjaga label
`is_gegenpressing` terpisah dari outcome `label_regain_5s`. Label `-1`
tetap disimpan sebagai kasus ragu dan tidak digunakan untuk training.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.research_readiness import (
    DEFAULT_MATCH_SPLIT,
    build_match_split_manifest,
    build_model_manifest,
    validate_model_manifest,
)

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EPISODE_PATH = PROCESSED_DIR / 'episode_manifest.csv'
ANNOTATION_PATH = PROCESSED_DIR / 'gegenpressing_annotation_manifest.csv'
MODEL_MANIFEST_PATH = PROCESSED_DIR / 'model_episode_manifest.csv'
MATCH_SPLIT_PATH = PROCESSED_DIR / 'match_split_manifest.csv'

In [2]:
episodes = pd.read_csv(EPISODE_PATH)
annotations = pd.read_csv(ANNOTATION_PATH)

model_manifest = build_model_manifest(
    episodes,
    annotations,
    DEFAULT_MATCH_SPLIT,
)
match_split_manifest = build_match_split_manifest(model_manifest)

model_manifest.to_csv(MODEL_MANIFEST_PATH, index=False)
match_split_manifest.to_csv(MATCH_SPLIT_PATH, index=False)

print('Strict-valid model episodes:', len(model_manifest))
print('Model manifest:', MODEL_MANIFEST_PATH)
print('Match split manifest:', MATCH_SPLIT_PATH)

Strict-valid model episodes: 1873
Model manifest: c:\VAEP-Track\data\processed\model_episode_manifest.csv
Match split manifest: c:\VAEP-Track\data\processed\match_split_manifest.csv


In [3]:
display(match_split_manifest)

split_summary = (
    model_manifest.groupby('split', observed=True)
    .agg(
        matches=('match_id', 'nunique'),
        strict_valid_episodes=('candidate_id', 'size'),
        quick_regain_positive=('label_regain_5s', 'sum'),
        annotated_usable=('eligible_for_detection_training', 'sum'),
    )
)
split_summary['quick_regain_negative'] = (
    split_summary['strict_valid_episodes']
    - split_summary['quick_regain_positive']
)
split_summary['episode_share_percent'] = (
    100 * split_summary['strict_valid_episodes']
    / split_summary['strict_valid_episodes'].sum()
).round(2)
display(split_summary)

,split,match_id,strict_valid_episodes,quick_regain_positive,quick_regain_negative,annotated_usable,annotated_uncertain
0,test,J03WPY,323,147,176,3,0
1,train,J03WMX,408,211,197,10,0
2,train,J03WN1,33,19,14,3,0
3,train,J03WOH,298,136,162,2,1
4,train,J03WOY,342,170,172,3,0
5,train,J03WR9,274,110,164,2,1
6,validation,J03WQQ,195,72,123,3,0


,matches,strict_valid_episodes,quick_regain_positive,annotated_usable,quick_regain_negative,episode_share_percent
split,,,,,,
test,1,323,147,3,176,17.25
train,5,1355,646,20,709,72.34
validation,1,195,72,3,123,10.41


In [4]:
pilot = annotations[
    annotations['pilot_selected'].astype(str).str.lower().eq('true')
    & annotations['is_gegenpressing'].notna()
].copy()
pilot['is_gegenpressing'] = pilot['is_gegenpressing'].astype(int)

print('Pilot annotations:', len(pilot))
display(
    pilot['is_gegenpressing']
    .value_counts()
    .sort_index()
    .rename('count')
    .to_frame()
)
print('Behavior label versus quick-regain outcome:')
display(pd.crosstab(
    pilot['is_gegenpressing'],
    pilot['quick_regain_success'],
    margins=True,
))
print('Usable manual labels by split:')
usable = model_manifest[model_manifest['eligible_for_detection_training']]
display(pd.crosstab(
    usable['split'],
    usable['gegenpressing_label'],
    margins=True,
))

Pilot annotations: 28


,count
is_gegenpressing,
-1,2
0,5
1,21


Behavior label versus quick-regain outcome:


quick_regain_success,0,1,All
is_gegenpressing,,,
-1,2,0,2
0,2,3,5
1,10,11,21
All,14,14,28


Usable manual labels by split:


gegenpressing_label,0,1,All
split,,,
test,1,2,3
train,3,17,20
validation,1,2,3
All,5,21,26


In [5]:
checks = validate_model_manifest(model_manifest)
display(checks.to_frame())
assert checks.all(), 'Ada pemeriksaan manifest yang gagal.'

print('Kesimpulan kesiapan:')
print('- Pembagian pertandingan bebas kebocoran antarsplit.')
print('- Outcome quick regain dan label gegenpressing tetap terpisah.')
print('- Label ragu dikeluarkan dari training.')
print('- Pilot cukup untuk kalibrasi prosedur, belum cukup untuk training final.')

,passed
candidate_id_unique,True
all_matches_have_one_split,True
required_splits_present,True
only_binary_labels_trainable,True
uncertain_labels_excluded,True


Kesimpulan kesiapan:
- Pembagian pertandingan bebas kebocoran antarsplit.
- Outcome quick regain dan label gegenpressing tetap terpisah.
- Label ragu dikeluarkan dari training.
- Pilot cukup untuk kalibrasi prosedur, belum cukup untuk training final.
